# 09 — Heterogeneous-chemistry (Ivatt et al. 2022) test, saturating functional form

Decisive test for the 2026-09-11 TODO: Ivatt, Evans & Lewis (2022, Nat. Geosci.)-style
HOx/ROx radical uptake on aerosol surface area is mechanistically distinct from both
optical shading (ruled out) and the linear lagged interaction already tested null
(`pm25_lag1*no_ratio`). This lets PySR search freely over `pm25`, `photochemical_potential`
(=no2*radiation_mean, the NOx+sunlight production term radical uptake would suppress),
and `no_ratio` (titration), with division available so a saturating suppression form
(`photochemical_potential/(1+k*pm25)`-style) can emerge on its own if the data supports
it -- not just the one hand-picked k-sweep already run.


In [1]:
import sys, json
from pathlib import Path

PROJECT_ROOT = Path("/path/to/project")
sys.path.insert(0, str(PROJECT_ROOT / "Scripts"))

import numpy as np
import pandas as pd
from sklearn.metrics import r2_score

DATA_DIR = PROJECT_ROOT / "Data" / "processed" / "colombia" / "bogota"

df = pd.read_parquet(DATA_DIR / "bogota_o3_master_dataset.parquet")
df = df.dropna(subset=['o3']).reset_index(drop=True)
print(df.shape)


(3285, 219)


## Free search: `pm25`, `photochemical_potential`, `no_ratio` -> `o3`

In [2]:
from pysr import PySRRegressor

features = ['pm25', 'photochemical_potential', 'no_ratio']
data = df.dropna(subset=features + ['o3']).reset_index(drop=True)
X = data[features].to_numpy()
y = data['o3'].to_numpy()
print(data.shape)

sr = PySRRegressor(
    niterations=150,
    binary_operators=["+", "-", "*", "/"],
    unary_operators=["exp", "log", "sqrt"],
    model_selection="best",
    maxsize=22,
    populations=24,
)
sr.fit(X, y, variable_names=features)
print(sr)


Compiling Julia backend...


(2920, 219)


[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:



Expressions evaluated per second: 2.600e+04
Progress: 63 / 150 total iterations (42.500%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.892e+01  0.000e+00  y = 20.147
3           4.384e+01  5.479e-02  y = no_ratio + 18.886
4           3.690e+01  1.722e-01  y = sqrt(photochemical_potential) / 3.1369
5           3.671e+01  5.315e-03  y = sqrt(sqrt(photochemical_potential)) * 2.589
6           2.575e+01  3.546e-01  y = log(no_ratio * pm25) / 0.14069
7           2.565e+01  3.978e-03  y = (log(no_ratio) + log(pm25)) / 0.14381
9           2.458e+01  2.125e-02  y = sqrt(sqrt(pm25 * ((photochemical_potential + 4405.6) *...
                                       no_ratio)))
11          2.453e+01  1.023e-03  y = sqrt(sqrt((photochemical_potential - -6084.2) * (no_ra...
                 

PySRRegressor.equations_ = [
	    pick     score                                           equation  \
	0         0.000000                                          20.146914   
	1         0.054788                               no_ratio + 18.885223   
	2         0.172217           sqrt(photochemical_potential) / 3.136942   
	3         0.013777                   sqrt(sqrt(no_ratio * 151642.36))   
	4   >>>>  0.350628                  log(no_ratio * pm25) / 0.14292754   
	5         0.004345    log((no_ratio * pm25) + 1.1593982) / 0.14611056   
	6         0.033867  sqrt(sqrt(pm25 * (no_ratio * (photochemical_po...   
	7         0.002715  sqrt(sqrt((photochemical_potential - -6686.706...   
	8         0.008074  sqrt(sqrt((pm25 * no_ratio) * (sqrt(pm25 * 1.1...   
	9         0.000580  sqrt(sqrt((((1027.8446 * sqrt(pm25)) + photoch...   
	10        0.000181  sqrt(sqrt(no_ratio * ((photochemical_potential...   
	11        0.000032  -0.19940966 + sqrt(sqrt(no_ratio * ((photochem...   
	12      

In [3]:
pareto = sr.equations_[['complexity', 'loss', 'equation']]
pareto['r2'] = pareto.apply(lambda row: r2_score(y, sr.predict(X, index=row.name)), axis=1)
pareto


,complexity,loss,equation,r2
0,1,48.916985,20.146914,-1.891777e-07
1,3,43.840107,no_ratio + 18.885223,1.037854e-01
2,4,36.904450,sqrt(photochemical_potential) / 3.136942,2.455697e-01
3,5,36.399500,sqrt(sqrt(no_ratio * 151642.36)),2.558922e-01
4,6,25.634182,log(no_ratio * pm25) / 0.14292754,4.759655e-01
5,8,25.412409,log((no_ratio * pm25) + 1.1593982) / 0.14611056,4.804992e-01
6,9,24.566185,sqrt(sqrt(pm25 * (no_ratio * (photochemical_po...,4.977984e-01
7,11,24.433134,sqrt(sqrt((photochemical_potential - -6686.706...,5.005183e-01
8,12,24.236649,sqrt(sqrt((pm25 * no_ratio) * (sqrt(pm25 * 1.1...,5.045350e-01
9,14,24.208540,sqrt(sqrt((((1027.8446 * sqrt(pm25)) + photoch...,5.051097e-01


## Interpretation

Check whether any equation on the Pareto front has `pm25` appearing as a SUPPRESSING
denominator term on `photochemical_potential` (i.e. `photochemical_potential / (something
increasing in pm25)`) -- that would be positive evidence for the heterogeneous-chemistry
suppression mechanism. If `pm25` only ever appears as a positive additive/multiplicative
term (as in every previous test this session), that's a further decisive negative result
for this mechanism, consistent with the k-sweep already run.


In [4]:
baseline_r2 = r2_score(y, np.polyval(np.polyfit(data['photochemical_potential'], y, 1), data['photochemical_potential']))
print(f"photochemical_potential-only linear baseline R2: {baseline_r2:.3f}")
print()
print("Best (highest-R2) equation:")
print(pareto.loc[pareto['r2'].idxmax()])
print()
print("Full Pareto front:")
for _, row in pareto.sort_values('complexity').iterrows():
    print(f"  [{int(row['complexity']):2d}] R2={row['r2']:.3f}  {row['equation']}")


photochemical_potential-only linear baseline R2: 0.264

Best (highest-R2) equation:
complexity                                                   22
loss                                                  24.134373
equation      log(no_ratio * (photochemical_potential + no_r...
r2                                                     0.506626
Name: 13, dtype: object

Full Pareto front:
  [ 1] R2=-0.000  20.146914
  [ 3] R2=0.104  no_ratio + 18.885223
  [ 4] R2=0.246  sqrt(photochemical_potential) / 3.136942
  [ 5] R2=0.256  sqrt(sqrt(no_ratio * 151642.36))
  [ 6] R2=0.476  log(no_ratio * pm25) / 0.14292754
  [ 8] R2=0.480  log((no_ratio * pm25) + 1.1593982) / 0.14611056
  [ 9] R2=0.498  sqrt(sqrt(pm25 * (no_ratio * (photochemical_potential + 4593.132))))
  [11] R2=0.501  sqrt(sqrt((photochemical_potential - -6686.706) * (pm25 * no_ratio)) - 42.055073)
  [12] R2=0.505  sqrt(sqrt((pm25 * no_ratio) * (sqrt(pm25 * 1.1701339e6) + photochemical_potential)))
  [14] R2=0.505  sqrt(sqrt((((1027.8446

## Save

In [5]:
pareto.to_csv(DATA_DIR / "bogota_o3_heterogeneous_chem_test_pareto.csv", index=False)
print("Saved")


Saved
